In [8]:
import pandas as pd
import plotly.express as px

In [9]:
# ROLL: Data Loading (Andmete laadimine) -- Krista

# sales andmete laadimine.
df_sales = pd.read_csv('sales.csv')
# customers andmete laadimine.
df_customers = pd.read_csv('customers.csv')

# Tabelite ühendamine
df_merged = pd.merge(df_sales, df_customers, on='customer_id', how='left')                    ## Ühendame müügi- ja klienditabelid customer_id

# Kontrollime, et kõik vajalikud veerud on olemas
required_cols = ['customer_id', 'sale_date', 'total_price', 'email']
print("Nõutud veerud olemas:", all(c in df_merged.columns for c in required_cols))

# Kontrollime ühendatud tabeli struktuuri
print("\n--- TABELITE STRUKTUUR ---")
# Kuvab andmete ridade ja veergude arvu
print(df_merged.shape)
# kuvab esimesed 5 elementi tabelist
print(df_merged.head())
# kuvab tabelite andmetüüpe
print(df_merged.dtypes)

Nõutud veerud olemas: True

--- TABELITE STRUKTUUR ---
(10118, 19)
   sale_id        invoice_id   sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001  2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002  2023-01-16       4338.0        1207         2   
2        3  INV-202301-00003  2023-01-05       4673.0        1264         1   
3        4  INV-202301-00004  2023-01-02       4677.0        1341         3   
4        5  INV-202301-00005  2023-01-13       2390.0        1284         1   

   unit_price  total_price channel store_location payment_method first_name  \
0      234.79       469.58    pood        Tallinn          kaart      Hille   
1      241.13       482.26    pood          Pärnu      järelmaks      Merle   
2      258.46       221.19    pood          Pärnu      järelmaks      Liina   
3       45.21       135.63    pood          Tartu       sularaha       Aili   
4       99.57        99.57    pood          Tartu          kaar

In [10]:
# ROLL: Data Cleaning (Andmete puhastamine) -- Egle

# Kuvame algset ridade arvu
print("Algne ridade arv:", df_merged.shape)

# Duplikaatide kontroll ja eemaldamine
print("Duplikaatide arv:", df_merged.duplicated().sum())
df_cleaned = df_merged.drop_duplicates()

# NULL väärtuste kontroll ja kriitiliste ridade eemaldamine
print("NULL väärtused veergudes:\n", df_cleaned.isnull().sum())

# Eemaldame read, kus puudub ID, kuupäev või hind
df_cleaned = df_cleaned.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Kuupäevade muutmine õigesse formaati
df_cleaned['sale_date'] = pd.to_datetime(df_cleaned['sale_date'])

# Eemaldame müügid, mis on hilisemad kui viitekuupäev
df_cleaned = df_cleaned[df_cleaned['sale_date'] <= pd.to_datetime('2025-02-28')]

# Vigaste väärtuste eemaldamine
# Jätame alles ainult need read, kus hind on suurem kui 0
df_cleaned = df_cleaned[df_cleaned['total_price'] > 0]

# Kuvame puhastusraporti, et veenduda andmestiku korrektsuses
print("\n--- PUHASTUSRAPORT ---")
print("Lõplik ridade arv (shape):", df_cleaned.shape)
print("Unikaalseid kliente:", df_cleaned['customer_id'].nunique())
print("Periood:", df_cleaned['sale_date'].min(), "kuni", df_cleaned['sale_date'].max())

# Kuvame puhastatud andmete alguse
print("\nValmis andmed analüüsiks:")
print(df_cleaned.head())

Algne ridade arv: (10118, 19)
Duplikaatide arv: 0
NULL väärtused veergudes:
 sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

--- PUHASTUSRAPORT ---
Lõplik ridade arv (shape): (8923, 19)
Unikaalseid kliente: 2540
Periood: 2023-01-01 00:00:00 kuni 2025-02-28 00:00:00

Valmis andmed analüüsiks:
   sale_id        invoice_id  sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001 2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002 2023-01-16       4338.0        1207         2   
2        3  I

In [14]:
# ROLL: Analysis — RFM kliendisegmenteerimine -- Kevin

# Viitekuupäev on fikseeritud, et tulemused oleksid võrreldavad
today = pd.to_datetime('2025-02-28')

# Samm 2: Arvuta Recency — viimane ostu kuupäev ja päevade arv
recency_df = df_cleaned.groupby('customer_id')['sale_date'].max().reset_index()
recency_df.columns = ['customer_id', 'last_purchase']
recency_df['recency_days'] = (today - recency_df['last_purchase']).dt.days

# Samm 3: Arvuta Frequency — ostude arv
frequency_df = df_cleaned.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# Samm 4: Arvuta Monetary — kogukulutus
monetary_df = df_cleaned.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary']

# Samm 5: Liida R, F, M ühte tabelisse pd.merge abil
rfm = pd.merge(recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id')
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# Määra skoorid 1-5 (pd.qcut jagab andmed 5 võrdsesse gruppi)
# Recency puhul on väiksem number parem, seega sildid on tagurpidi [5, 4, 3, 2, 1]
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

# Samm 6: Arvuta summaarne RFM skoor
rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Segmenteerimisfunktsioon skooride põhjal
def määra_segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(määra_segment)

# VÄLJUND: RFM kokkuvõttetabel
print("--- RFM SEGMENTIDE KOKKUVÕTE ---")
segmentide_arv = rfm['Segment'].value_counts()
segmentide_protsent = rfm['Segment'].value_counts(normalize=True) * 100

kokkuvotte_df = pd.DataFrame({
    'Klientide arv': segmentide_arv,
    'Osakaal (%)': segmentide_protsent.round(1)
})

print(kokkuvotte_df)

# Kontroll: kas kõik kliendid said segmendi?
print("\nSegmendita kliente:", rfm['Segment'].isnull().sum())

# Kuvame tabeli alguse, et näha tulemusi
print("\nNäidis kliendiandmetest koos skooridega:")
rfm.head()

--- RFM SEGMENTIDE KOKKUVÕTE ---
               Klientide arv  Osakaal (%)
Segment                                  
Potential                768         30.2
Loyal                    678         26.7
At Risk                  524         20.6
VIP Champions            453         17.8
Lost                     117          4.6

Segmendita kliente: 0

Näidis kliendiandmetest koos skooridega:


,customer_id,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,Segment
0,2001.0,91,2,203.92,4,1,1,6,At Risk
1,2004.0,71,2,1198.56,4,1,4,9,Potential
2,2005.0,148,4,959.60,3,4,4,11,Loyal
3,2006.0,477,1,327.06,1,1,1,3,Lost
4,2007.0,29,1,318.63,5,1,1,7,Potential


In [12]:
# ROLL: Visualization (Visualiseerimine ja leiud) -- Eike

# DIAGRAMM: Segmentide jaotus (Tulpdiagramm)
# Loeme kokku, mitu klienti on igas segmendis
segment_counts = rfm['Segment'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Klientide arv']

fig1 = px.bar(segment_counts, 
             x='Segment', 
             y='Klientide arv', 
             title='Kliendibaasi jaotus segmentide lõikes',
             color='Segment',
             text_auto=True)
fig1.show()

# DIAGRAMM: Klientide käitumine (Hajuvusdiagramm)
# x-telg: päevi viimasest ostust, y-telg: rahaline väärtus, mullikese suurus: ostusagedus
fig2 = px.scatter(rfm.reset_index(), 
                 x='recency', 
                 y='monetary',
                 size='frequency', 
                 color='Segment',
                 hover_name='customer_id',
                 title='Klientide väärtus vs. värskus (Mulli suurus = ostude arv)',
                 labels={'recency': 'Päevi viimasest ostust', 'monetary': 'Kogukulu (€)'})
fig2.show()

# DIAGRAMM: Top 10 VIP klienti
top_vips = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary')

fig3 = px.bar(top_vips, 
             x=top_vips.index.astype(str), 
             y='monetary',
             title='Top 10 VIP klienti kogukulutuse järgi',
             labels={'x': 'Kliendi ID', 'monetary': 'Kogukulu (€)'},
             color_discrete_sequence=['#FFD700']) # Kuldne värv VIP-dele
fig3.show()

# --- ÄRITÕLGENDUS MARKOLLE ---
print("\n--- KOKKUVÕTE MARKOLLE ---")
vip_count = len(rfm[rfm['Segment'] == 'VIP Champions'])
at_risk_count = len(rfm[rfm['Segment'] == 'At Risk'])
total_revenue = rfm['monetary'].sum()
vip_revenue = rfm[rfm['Segment'] == 'VIP Champions']['monetary'].sum()
vip_share = (vip_revenue / total_revenue) * 100

print(f"Meie andmetes on {vip_count} VIP-klienti, kes moodustavad tervelt {vip_share:.1f}% kogu käibest.")
print(f"Samas on meil {at_risk_count} klienti staatuses 'At Risk', kes pole ammu ostnud.")
print("Peamine fookus peaks olema VIP-ide hoidmisel ja ootel olevate klientide reaktiveerimisel.")

# --- KONKREETSED SOOVITUSED ---
print("\n--- SOOVITUSED ---")
print("1. VIP PROGRAMM: Pakkuda top 10 kliendile personaalset teenindust ja varajast ligipääsu uutele toodetele.")
print("2. WIN-BACK KAMPAANIA: Saata 'At Risk' segmendile sooduskood, et motiveerida neid uut ostu tegema.")
print("3. NURTURE PROGRAMM: 'Potential' segmendile suunata sisuturundust, et tõsta nende ostusagedust.")
vip_share = (vip_revenue / total_revenue) * 100

print(f"Meie andmetes on {vip_count} VIP-klienti, kes moodustavad tervelt {vip_share:.1f}% kogu käibest.")
print(f"Samas on meil {at_risk_count} klienti staatuses 'At Risk', kes pole ammu ostnud.")
print("Peamine fookus peaks olema VIP-ide hoidmisel ja ootel olevate klientide reaktiveerimisel.")

# --- KONKREETSED SOOVITUSED ---
print("\n--- SOOVITUSED ---")
print("1. VIP PROGRAMM: Pakkuda top 10 kliendile personaalset teenindust ja varajast ligipääsu uutele toodetele.")
print("2. WIN-BACK KAMPAANIA: Saata 'At Risk' segmendile sooduskood, et motiveerida neid uut ostu tegema.")
print("3. NURTURE PROGRAMM: 'Potential' segmendile suunata sisuturundust, et tõsta nende ostusagedust.")


--- KOKKUVÕTE MARKOLLE ---
Meie andmetes on 453 VIP-klienti, kes moodustavad tervelt 42.8% kogu käibest.
Samas on meil 524 klienti staatuses 'At Risk', kes pole ammu ostnud.
Peamine fookus peaks olema VIP-ide hoidmisel ja ootel olevate klientide reaktiveerimisel.

--- SOOVITUSED ---
1. VIP PROGRAMM: Pakkuda top 10 kliendile personaalset teenindust ja varajast ligipääsu uutele toodetele.
2. WIN-BACK KAMPAANIA: Saata 'At Risk' segmendile sooduskood, et motiveerida neid uut ostu tegema.
3. NURTURE PROGRAMM: 'Potential' segmendile suunata sisuturundust, et tõsta nende ostusagedust.
Meie andmetes on 453 VIP-klienti, kes moodustavad tervelt 42.8% kogu käibest.
Samas on meil 524 klienti staatuses 'At Risk', kes pole ammu ostnud.
Peamine fookus peaks olema VIP-ide hoidmisel ja ootel olevate klientide reaktiveerimisel.

--- SOOVITUSED ---
1. VIP PROGRAMM: Pakkuda top 10 kliendile personaalset teenindust ja varajast ligipääsu uutele toodetele.
2. WIN-BACK KAMPAANIA: Saata 'At Risk' segmendile s